# Feature Engineering

---

1. Import packages
2. Load data
3. Feature engineering

---

## 1. Import packages

In [6]:
import pandas as pd
import datetime
import numpy as np

---
## 2. Load data

In [7]:
df = pd.read_csv('./clean_data_after_eda.csv')
df["date_activ"] = pd.to_datetime(df["date_activ"], format='%Y-%m-%d')
df["date_end"] = pd.to_datetime(df["date_end"], format='%Y-%m-%d')
df["date_modif_prod"] = pd.to_datetime(df["date_modif_prod"], format='%Y-%m-%d')
df["date_renewal"] = pd.to_datetime(df["date_renewal"], format='%Y-%m-%d')

In [8]:
df.head(3)

,id,channel_sales,cons_12m,cons_gas_12m,cons_last_month,date_activ,date_end,date_modif_prod,date_renewal,forecast_cons_12m,...,margin_gross_pow_ele,margin_net_pow_ele,nb_prod_act,net_margin,num_years_antig,origin_up,pow_max,churn,Churn,tenure_bucket
0,24011ae4ebbe3035111d65fa7c15bc57,foosdfpfkusacimwkcsosbicdxkicaua,0,54946,0,2013-06-15,2016-06-15,2015-11-01,2015-06-23,0.00,...,25.44,25.44,2,678.99,3,lxidpiddsbxsbosboudacockeimpuepw,43.648,1,Churned,<5 years
1,d29c2c54acc38ff3c0614d0a653813dd,MISSING,4660,0,0,2009-08-21,2016-08-30,2009-08-21,2015-08-31,189.95,...,16.38,16.38,1,18.89,6,kamkkxfxxuwbdslkwifmmcsiusiuosws,13.800,0,Retained,5-10 years
2,764c75f661154dac3a6c254cd082ea7d,foosdfpfkusacimwkcsosbicdxkicaua,544,0,0,2010-04-16,2016-04-16,2010-04-16,2015-04-17,47.96,...,28.60,28.60,1,6.60,6,kamkkxfxxuwbdslkwifmmcsiusiuosws,13.856,0,Retained,5-10 years


---

## 3. Feature engineering

### 1) Difference between off-peak prices in December and preceding January
price_df - dataframe for price data from price_data.csv file

In [9]:
price_df = pd.read_csv('./price_data.csv')
price_df.head()

,id,price_date,price_off_peak_var,price_peak_var,price_mid_peak_var,price_off_peak_fix,price_peak_fix,price_mid_peak_fix
0,038af19179925da21a25619c5a24b745,2015-01-01,0.151367,0.0,0.0,44.266931,0.0,0.0
1,038af19179925da21a25619c5a24b745,2015-02-01,0.151367,0.0,0.0,44.266931,0.0,0.0
2,038af19179925da21a25619c5a24b745,2015-03-01,0.151367,0.0,0.0,44.266931,0.0,0.0
3,038af19179925da21a25619c5a24b745,2015-04-01,0.149626,0.0,0.0,44.266931,0.0,0.0
4,038af19179925da21a25619c5a24b745,2015-05-01,0.149626,0.0,0.0,44.266931,0.0,0.0


In [10]:
price_df["price_date"] = pd.to_datetime(price_df["price_date"], format='%Y-%m-%d')
price_df.head()

,id,price_date,price_off_peak_var,price_peak_var,price_mid_peak_var,price_off_peak_fix,price_peak_fix,price_mid_peak_fix
0,038af19179925da21a25619c5a24b745,2015-01-01,0.151367,0.0,0.0,44.266931,0.0,0.0
1,038af19179925da21a25619c5a24b745,2015-02-01,0.151367,0.0,0.0,44.266931,0.0,0.0
2,038af19179925da21a25619c5a24b745,2015-03-01,0.151367,0.0,0.0,44.266931,0.0,0.0
3,038af19179925da21a25619c5a24b745,2015-04-01,0.149626,0.0,0.0,44.266931,0.0,0.0
4,038af19179925da21a25619c5a24b745,2015-05-01,0.149626,0.0,0.0,44.266931,0.0,0.0


In [11]:
# Group off-peak prices by companies and month
monthly_price_by_id = price_df.groupby(['id', 'price_date']).agg({'price_off_peak_var': 'mean', 'price_off_peak_fix': 'mean'}).reset_index()

# Get january and december prices
jan_prices = monthly_price_by_id.groupby('id').first().reset_index()
dec_prices = monthly_price_by_id.groupby('id').last().reset_index()

# Calculate the difference
diff = pd.merge(dec_prices.rename(columns={'price_off_peak_var': 'dec_1', 'price_off_peak_fix': 'dec_2'}), jan_prices.drop(columns='price_date'), on='id')
diff['offpeak_diff_dec_january_energy'] = diff['dec_1'] - diff['price_off_peak_var']
diff['offpeak_diff_dec_january_power'] = diff['dec_2'] - diff['price_off_peak_fix']
diff = diff[['id', 'offpeak_diff_dec_january_energy','offpeak_diff_dec_january_power']]
diff.head()

,id,offpeak_diff_dec_january_energy,offpeak_diff_dec_january_power
0,0002203ffbb812588b632b9e628cc38d,-0.006192,0.162916
1,0004351ebdd665e6ee664792efc4fd13,-0.004104,0.177779
2,0010bcc39e42b3c2131ed2ce55246e3c,0.050443,1.500000
3,0010ee3855fdea87602a5b7aba8e42de,-0.010018,0.162916
4,00114d74e963e47177db89bc70108537,-0.003994,-0.000001


In [12]:
# merging price data features with client dataframe
df = pd.merge(df, diff, on='id')
print(df.shape)

(14606, 30)


### 2) Handling highly correlated 'margin' columns
df - dataframe for client data from the main file - clean_data_after_eda.csv

In [13]:
df.shape

(14606, 30)

In [14]:
df.columns

Index(['id', 'channel_sales', 'cons_12m', 'cons_gas_12m', 'cons_last_month',
       'date_activ', 'date_end', 'date_modif_prod', 'date_renewal',
       'forecast_cons_12m', 'forecast_cons_year', 'forecast_discount_energy',
       'forecast_meter_rent_12m', 'forecast_price_energy_off_peak',
       'forecast_price_energy_peak', 'forecast_price_pow_off_peak', 'has_gas',
       'imp_cons', 'margin_gross_pow_ele', 'margin_net_pow_ele', 'nb_prod_act',
       'net_margin', 'num_years_antig', 'origin_up', 'pow_max', 'churn',
       'Churn', 'tenure_bucket', 'offpeak_diff_dec_january_energy',
       'offpeak_diff_dec_january_power'],
      dtype='object')

In [15]:
# checking the correlation between margin columns that look identical
df[['margin_gross_pow_ele', 'margin_net_pow_ele']].corr()

,margin_gross_pow_ele,margin_net_pow_ele
margin_gross_pow_ele,1.000000,0.999914
margin_net_pow_ele,0.999914,1.000000


In [16]:
# checking zero-inflation in forecast dicount energy
df['forecast_discount_energy'].value_counts()

forecast_discount_energy
0.0     14094
30.0      260
28.0      102
24.0       83
22.0       47
25.0        7
26.0        5
19.0        2
17.0        2
23.0        2
10.0        1
5.0         1
Name: count, dtype: int64

In [17]:
df = df.drop(columns=['margin_gross_pow_ele','forecast_discount_energy'])
df.shape

(14606, 28)

### 3) Feature Engineering on Dates

In [18]:
reference_date = datetime.datetime(2016, 1, 1)

In [19]:
# Number of months since activation (tenure)
df['months_activ'] = ((reference_date - df['date_activ']).dt.days / 30).astype(int)

# Number of months until contract ends
df['months_to_end'] = ((df['date_end'] - reference_date).dt.days / 30).astype(int)

# Number of months since last modification
df['months_modif_prod'] = ((reference_date - df['date_modif_prod']).dt.days / 30).astype(int)

# Number of months since last renewal
df['months_renewal'] = ((reference_date - df['date_renewal']).dt.days / 30).astype(int)

In [20]:
df[['months_activ', 'months_to_end', 'months_modif_prod', 'months_renewal']].head()
df.shape

(14606, 32)

In [21]:
# dropping the original date columns that the model cannot process
df = df.drop(columns=['date_activ', 'date_end', 'date_modif_prod', 'date_renewal'])
print(df.shape)

(14606, 28)


### 4) Handling highly correlated tenure columns - 'months_active' and 'num_years_antiq'

In [22]:
df[['months_activ', 'num_years_antig']].corr()

,months_activ,num_years_antig
months_activ,1.000000,0.984072
num_years_antig,0.984072,1.000000


In [23]:
# Dropping num_years_antig - highly correlated with months_activ
df = df.drop(columns=['num_years_antig'])

In [24]:
df.shape

(14606, 27)

### 5) Feature for Average price changes across periods
This feature carries the average price change for different companies across individual periods.

In [25]:
mean_prices = price_df.groupby(['id']).agg({
    'price_off_peak_var': 'mean', 
    'price_peak_var': 'mean', 
    'price_mid_peak_var': 'mean',
    'price_off_peak_fix': 'mean',
    'price_peak_fix': 'mean',
    'price_mid_peak_fix': 'mean'    
}).reset_index()

In [26]:
# Calculate the mean difference between consecutive periods
mean_prices['off_peak_peak_var_mean_diff'] = mean_prices['price_off_peak_var'] - mean_prices['price_peak_var']
mean_prices['peak_mid_peak_var_mean_diff'] = mean_prices['price_peak_var'] - mean_prices['price_mid_peak_var']
mean_prices['off_peak_mid_peak_var_mean_diff'] = mean_prices['price_off_peak_var'] - mean_prices['price_mid_peak_var']
mean_prices['off_peak_peak_fix_mean_diff'] = mean_prices['price_off_peak_fix'] - mean_prices['price_peak_fix']
mean_prices['peak_mid_peak_fix_mean_diff'] = mean_prices['price_peak_fix'] - mean_prices['price_mid_peak_fix']
mean_prices['off_peak_mid_peak_fix_mean_diff'] = mean_prices['price_off_peak_fix'] - mean_prices['price_mid_peak_fix']

In [27]:
# merging the features to client data
columns = [
    'id', 
    'off_peak_peak_var_mean_diff',
    'peak_mid_peak_var_mean_diff', 
    'off_peak_mid_peak_var_mean_diff',
    'off_peak_peak_fix_mean_diff', 
    'peak_mid_peak_fix_mean_diff', 
    'off_peak_mid_peak_fix_mean_diff'
]
df = pd.merge(df, mean_prices[columns], on='id')
df.head()

,id,channel_sales,cons_12m,cons_gas_12m,cons_last_month,forecast_cons_12m,forecast_cons_year,forecast_meter_rent_12m,forecast_price_energy_off_peak,forecast_price_energy_peak,...,months_activ,months_to_end,months_modif_prod,months_renewal,off_peak_peak_var_mean_diff,peak_mid_peak_var_mean_diff,off_peak_mid_peak_var_mean_diff,off_peak_peak_fix_mean_diff,peak_mid_peak_fix_mean_diff,off_peak_mid_peak_fix_mean_diff
0,24011ae4ebbe3035111d65fa7c15bc57,foosdfpfkusacimwkcsosbicdxkicaua,0,54946,0,0.00,0,1.78,0.114481,0.098142,...,31,5,2,6,0.024038,0.034219,0.058257,18.590255,7.450670,26.040925
1,d29c2c54acc38ff3c0614d0a653813dd,MISSING,4660,0,0,189.95,0,16.27,0.145711,0.000000,...,77,8,77,4,0.142485,0.007124,0.149609,44.311375,0.000000,44.311375
2,764c75f661154dac3a6c254cd082ea7d,foosdfpfkusacimwkcsosbicdxkicaua,544,0,0,47.96,0,38.72,0.165794,0.087899,...,69,3,69,8,0.082090,0.088421,0.170512,44.385450,0.000000,44.385450
3,bba03439a292a1e166f80264c16191cb,lmkebamcaaclubfxadlmueccxoimlema,1584,0,0,240.04,0,19.83,0.146694,0.000000,...,70,2,70,9,0.151210,0.000000,0.151210,44.400265,0.000000,44.400265
4,149d57cf92fc41cf94415803a877cb4b,MISSING,4425,0,526,445.75,526,131.73,0.116900,0.100015,...,72,2,72,9,0.020536,0.030773,0.051309,16.275263,8.137629,24.412893


This feature reflects average price changes between consecutive periods (peak, mid-peak, 0ff-peak) for client companies. While the previous feature captured price changes across entire year (December and preceding January), this feature adopts a more granular approach and is useful to reveal patterns on a micro-scale between consecutive periods.

### 6) Encoding categorical variables

In [28]:
# replacing 't' and 'f' in has_gas with 1,0
df['has_gas'] = df['has_gas'].map({'t': 1, 'f': 0})
df['has_gas'].value_counts()

has_gas
0    11955
1     2651
Name: count, dtype: int64

In [29]:
# making dummies for channel_sales and origin_up columns
df = pd.get_dummies(df, columns=['channel_sales', 'origin_up'], drop_first=True, dtype=int)
print(df.shape)

(14606, 43)


### 7) Log transformation of skewed columns

In [30]:
skewed_cols = ['cons_12m', 'cons_gas_12m', 'cons_last_month', 
               'forecast_cons_12m', 'forecast_cons_year', 
               'forecast_meter_rent_12m', 'imp_cons', 'pow_max']

for col in skewed_cols:
    df[col] = np.log1p(df[col])

In [31]:
# checking skewness after tranformation
skewed_cols = ['cons_12m', 'cons_gas_12m', 'cons_last_month', 
               'forecast_cons_12m', 'forecast_cons_year', 
               'forecast_meter_rent_12m', 'imp_cons', 'pow_max']

for col in skewed_cols:
    print(f"{col}: {df[col].skew():.4f}")

cons_12m: -0.3795
cons_gas_12m: 1.8757
cons_last_month: -0.1907
forecast_cons_12m: -2.0328
forecast_cons_year: -0.1186
forecast_meter_rent_12m: -0.6036
imp_cons: 0.0119
pow_max: 1.8045


### 8) Removing the id column and saving final dataset

In [32]:
df = df.drop(columns=['id'])
df.to_csv('./data_after_feature_engineering.csv', index=False)
print(df.shape)
print("Dataset saved successfully!")

(14606, 42)
Dataset saved successfully!
